# Física del Caos: El Desafío del Transporte Seguro 💧

¿Has notado qué pasa en la micro o en el metro cuando el chofer acelera de golpe? Te vas hacia atrás. ¿Y cuando frena bruscamente? Te vas hacia adelante. A esto se le llama **Inercia**.

En este laboratorio, tu robot se convertirá en un vehículo de transporte de carga delicada. Descubrirás por qué encender un motor al 100% de inmediato es una pésima idea en el mundo real.


In [10]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulador: Inercia y Aceleración (Bucle For)</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">
        Observa el <strong>Vector de Inercia (Naranja)</strong>. Cuando el robot acelera, la inercia empuja el agua hacia atrás. Ajusta el <code>Wait(ms)</code> para crear una aceleración suave. Si el tiempo es muy corto, el tirón inicial derramará el agua antes de que el robot pueda avanzar.
    </p>
    
    <div style="display: flex; gap: 20px; margin-bottom: 20px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 10px; align-items: center;">
            <button id="play_UID" style="background-color: #0d6efd; color: white; border: none; padding: 10px 15px; border-radius: 5px; cursor: pointer; font-weight: bold; font-size: 14px;">▶ Ejecutar Codigo NXC</button>
            <button id="reset_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 15px; border-radius: 5px; cursor: pointer; font-weight: bold; font-size: 14px;">🔄 Limpiar y Reiniciar</button>
        </div>
        
        <div style="display: flex; flex-direction: column; align-items: center; background: #fff; padding: 10px 20px; border-radius: 5px; border: 1px solid #ccc;">
            <label style="font-weight: 600; font-size: 0.85em; color: #6f42c1; margin-bottom: 5px;">Tiempo de incremento: Wait( ? )</label>
            <div style="display: flex; align-items: center; gap: 10px;">
                <input type="range" id="wait_val_UID" min="20" max="300" step="20" value="100" style="width: 150px; cursor: pointer;">
                <span id="lbl_wait_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; width: 65px; color: #212529;">100 ms</span>
            </div>
        </div>
    </div>

    <div style="display: flex; gap: 15px; justify-content: center; margin-bottom: 15px; flex-wrap: wrap;">
        <div style="background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc; width: 140px; text-align: center;">
            <label style="font-weight: 600; font-size: 0.8em; margin-bottom: 2px; color: #fd7e14; display: block;">Potencia Motor</label>
            <span id="tele_pot_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; color: #212529;">0 / 70</span>
        </div>
        <div style="background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc; width: 150px; text-align: center;">
            <label style="font-weight: 600; font-size: 0.8em; margin-bottom: 2px; color: #dc3545; display: block;">Aceleracion (Inercia)</label>
            <span id="tele_acc_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; color: #212529;">0.00 m/s²</span>
        </div>
        <div style="background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc; width: 180px; text-align: center;">
            <label style="font-weight: 600; font-size: 0.8em; margin-bottom: 2px; color: #0d6efd; display: block;">Estado de la Carga</label>
            <span id="tele_status_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; color: #212529;">Segura</span>
        </div>
    </div>
    
    <div style="display: flex; justify-content: center; margin-bottom: 10px;">
        <canvas id="canvas_sim_UID" width="800" height="300" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_sim_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_play = document.getElementById('play_UID');
            const btn_reset = document.getElementById('reset_UID');
            const slider_wait = document.getElementById('wait_val_UID');
            const lbl_wait = document.getElementById('lbl_wait_UID');
            
            const tele_pot = document.getElementById('tele_pot_UID');
            const tele_acc = document.getElementById('tele_acc_UID');
            const tele_status = document.getElementById('tele_status_UID');
            
            // Constantes Visuales
            const Y_FLOOR = 240;
            const START_X = 50;
            const END_X = 700;
            
            // Estado NXC
            let isRunning = false;
            let waitMs = parseInt(slider_wait.value);
            let nxcPhase = 0; 
            let power = 0;
            let nxcTimer = 0;
            
            // Físicas del Robot
            let lastTime = 0;
            let robotX = START_X;
            let velocity = 0;
            let accel = 0;
            let actualMotorPower = 0; 
            
            const mass = 45; 
            const dragCoeff = 35; 
            const forceMultiplier = 100; 
            
            // Físicas del Agua
            let waterAngle = 0;
            let waterAngularVel = 0;
            const MAX_ANGLE = 0.38; 
            let isSpilled = false;
            let particles = [];

            function drawArrow(x, y, angle, length, color) {
                ctx.save();
                ctx.translate(x, y);
                ctx.rotate(angle);
                ctx.strokeStyle = color;
                ctx.fillStyle = color;
                ctx.lineWidth = 4;
                ctx.beginPath();
                ctx.moveTo(0, 0);
                ctx.lineTo(length, 0);
                ctx.stroke();
                
                ctx.beginPath();
                ctx.moveTo(length + 8, 0);
                ctx.lineTo(length - 4, 6);
                ctx.lineTo(length - 4, -6);
                ctx.fill();
                ctx.restore();
            }

            function drawWaterGlass(x, y, wAngle, spilled) {
                const cupX = x + 35; 
                const cupY = y - 55; 
                
                ctx.save();
                ctx.translate(cupX, cupY);
                
                let accel_ms2 = accel / 100;
                if (!spilled && Math.abs(accel_ms2) > 0.1) {
                    let inertiaForce = -accel_ms2 * 30;
                    let arrowLen = Math.abs(inertiaForce);
                    let arrowAng = inertiaForce > 0 ? 0 : Math.PI;
                    if (arrowLen > 10) {
                        drawArrow(0, -70, arrowAng, arrowLen, "#fd7e14");
                        ctx.fillStyle = "#fd7e14";
                        ctx.font = "bold 12px sans-serif";
                        ctx.fillText("Inercia", -20, -85);
                    }
                }

                ctx.beginPath();
                ctx.moveTo(-20, 0); 
                ctx.lineTo(20, 0);  
                ctx.lineTo(25, -50); 
                ctx.lineTo(-25, -50); 
                ctx.closePath();
                
                ctx.save();
                ctx.clip();
                
                if (!spilled) {
                    ctx.translate(0, -25); 
                    ctx.rotate(wAngle);
                    ctx.fillStyle = "rgba(13, 110, 253, 0.75)";
                    ctx.fillRect(-40, 0, 80, 50); 
                } else {
                    ctx.fillStyle = "rgba(13, 110, 253, 0.2)";
                    ctx.fillRect(-25, -5, 50, 10);
                }
                ctx.restore(); 
                
                ctx.strokeStyle = "rgba(173, 216, 230, 0.9)";
                ctx.lineWidth = 3;
                ctx.beginPath();
                ctx.moveTo(-25, -50);
                ctx.lineTo(-20, 0);
                ctx.lineTo(20, 0);
                ctx.lineTo(25, -50);
                ctx.stroke();
                
                if (!spilled) {
                    ctx.strokeStyle = "rgba(220, 53, 69, 0.6)";
                    ctx.lineWidth = 2;
                    ctx.setLineDash([4, 4]);
                    ctx.beginPath(); ctx.moveTo(-25, -50); ctx.lineTo(0, -25); ctx.stroke();
                    ctx.beginPath(); ctx.moveTo(25, -50); ctx.lineTo(0, -25); ctx.stroke();
                    ctx.setLineDash([]);
                }
                
                ctx.restore();
            }

            function drawRobot(x, y) {
                ctx.fillStyle = "#333";
                ctx.fillRect(x + 5, y - 15, 20, 25);
                ctx.fillRect(x + 45, y - 15, 20, 25);
                
                ctx.fillStyle = "#f8f9fa";
                ctx.strokeStyle = "#adb5bd";
                ctx.lineWidth = 2;
                ctx.beginPath();
                ctx.roundRect(x, y - 50, 70, 45, 5);
                ctx.fill(); ctx.stroke();
                
                ctx.fillStyle = "#495057";
                ctx.fillRect(x - 5, y - 55, 80, 5);
                
                ctx.fillStyle = "#c2d1b2";
                ctx.fillRect(x + 15, y - 45, 25, 18);
            }

            function createSplash(x, y) {
                for (let i = 0; i < 30; i++) {
                    particles.push({
                        x: x,
                        y: y,
                        vx: (Math.random() - 0.5) * 200 + velocity * 0.8,
                        vy: (Math.random() * -150) - 50,
                        r: Math.random() * 4 + 2
                    });
                }
            }

            function drawSim() {
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                
                ctx.fillStyle = "#e9ecef";
                ctx.fillRect(0, Y_FLOOR, canvas.width, canvas.height - Y_FLOOR);
                ctx.strokeStyle = "#ced4da";
                ctx.lineWidth = 1;
                ctx.beginPath(); ctx.moveTo(0, Y_FLOOR); ctx.lineTo(canvas.width, Y_FLOOR); ctx.stroke();
                
                ctx.fillStyle = "#28a745";
                ctx.fillRect(END_X, Y_FLOOR - 80, 5, 80);
                ctx.beginPath(); ctx.moveTo(END_X + 5, Y_FLOOR - 80); ctx.lineTo(END_X + 35, Y_FLOOR - 65); ctx.lineTo(END_X + 5, Y_FLOOR - 50); ctx.fill();

                drawRobot(robotX, Y_FLOOR);
                drawWaterGlass(robotX, Y_FLOOR, waterAngle, isSpilled);
                
                ctx.fillStyle = "#0d6efd";
                for (let i = 0; i < particles.length; i++) {
                    ctx.beginPath(); ctx.arc(particles[i].x, particles[i].y, particles[i].r, 0, Math.PI * 2); ctx.fill();
                }
                
                // Textos planos sin caracteres especiales
                if (isSpilled) {
                    ctx.fillStyle = "#dc3545"; ctx.font = "bold 26px sans-serif";
                    ctx.fillText("Carga Derramada!", canvas.width/2 - 120, 50);
                } else if (nxcPhase === 4) {
                    ctx.fillStyle = "#198754"; ctx.font = "bold 26px sans-serif";
                    ctx.fillText("Transporte Exitoso!", canvas.width/2 - 130, 50);
                }

                lbl_wait.innerText = slider_wait.value + " ms";
                tele_pot.innerText = power + " / 70";
                
                let accel_ms2 = accel / 100;
                let inertiaVal = Math.abs(accel_ms2);
                tele_acc.innerText = inertiaVal.toFixed(2) + " m/s²";
                
                if (inertiaVal > 1.0) tele_acc.style.color = "#dc3545"; 
                else if (inertiaVal > 0.5) tele_acc.style.color = "#fd7e14"; 
                else tele_acc.style.color = "#212529"; 

                // Estados sanitizados
                if (isSpilled) {
                    tele_status.innerText = "DERRAMADO!"; tele_status.style.color = "#dc3545";
                } else if (nxcPhase === 4) {
                    tele_status.innerText = "Llegada Segura"; tele_status.style.color = "#198754";
                } else if (isRunning) {
                    if (Math.abs(waterAngle) > 0.25) { 
                        tele_status.innerText = "PELIGRO!"; tele_status.style.color = "#fd7e14";
                    } else {
                        tele_status.innerText = "En movimiento"; tele_status.style.color = "#0d6efd";
                    }
                } else {
                    tele_status.innerText = "Esperando"; tele_status.style.color = "#6c757d";
                }
            }

            function updatePhysics(dt) {
                actualMotorPower += (power - actualMotorPower) * 10 * dt;

                let motorForce = actualMotorPower * forceMultiplier;
                let dragForce = velocity * dragCoeff;
                accel = (motorForce - dragForce) / mass;
                
                if (power === 0 && velocity < 1) { velocity = 0; accel = 0; }
                
                velocity += accel * dt;
                robotX += velocity * dt;
                
                if (robotX > END_X) {
                    robotX = END_X; velocity = 0; accel = 0;
                    if (isRunning) { power = 0; nxcPhase = 4; isRunning = false; }
                }

                if (!isSpilled) {
                    let accel_ms2 = accel / 100;
                    let targetAngle = -accel_ms2 * 0.45; 
                    
                    let restoringForce = (targetAngle - waterAngle) * 35; 
                    let dampingForce = -waterAngularVel * 3.5; 
                    
                    waterAngularVel += (restoringForce + dampingForce) * dt;
                    waterAngle += waterAngularVel * dt;
                    
                    if (Math.abs(waterAngle) > MAX_ANGLE) {
                        isSpilled = true;
                        createSplash(robotX + 35, Y_FLOOR - 55);
                        power = 0; 
                        isRunning = false;
                        btn_play.innerText = "▶ Codigo NXC (Fallo)";
                        btn_play.style.backgroundColor = "#dc3545";
                    }
                } else {
                    for (let i = particles.length - 1; i >= 0; i--) {
                        let p = particles[i];
                        p.vy += 600 * dt; 
                        p.x += p.vx * dt;
                        p.y += p.vy * dt;
                        if (p.y > Y_FLOOR) { p.y = Y_FLOOR; p.vx *= 0.5; }
                    }
                }
            }

            function animate(timestamp) {
                if (!lastTime) lastTime = timestamp;
                let dt = (timestamp - lastTime) / 1000;
                lastTime = timestamp;
                if (dt > 0.05) dt = 0.05; 
                
                if (isRunning && !isSpilled) {
                    nxcTimer += dt;
                    let waitSecs = waitMs / 1000;
                    
                    if (nxcPhase === 1) { 
                        if (nxcTimer >= waitSecs) {
                            power += 5; nxcTimer = 0;
                            if (power >= 70) { power = 70; nxcPhase = 2; nxcTimer = 0; }
                        }
                    } else if (nxcPhase === 2) { 
                        if (nxcTimer >= 2.0) { nxcPhase = 3; nxcTimer = 0; }
                    } else if (nxcPhase === 3) { 
                        if (nxcTimer >= waitSecs) {
                            power -= 5; nxcTimer = 0;
                            if (power <= 0) {
                                power = 0; nxcPhase = 4; isRunning = false;
                                btn_play.innerText = "▶ Ejecutar Codigo NXC";
                            }
                        }
                    }
                }
                
                updatePhysics(dt);
                drawSim();
                requestAnimationFrame(animate);
            }

            btn_play.addEventListener('click', () => {
                if (isSpilled || nxcPhase === 4) return;
                
                if (!isRunning) {
                    waitMs = parseInt(slider_wait.value);
                    isRunning = true;
                    if (nxcPhase === 0) nxcPhase = 1;
                    btn_play.innerText = "⏸ Ejecutando Bucle...";
                    btn_play.style.backgroundColor = "#ffc107";
                    btn_play.style.color = "#000";
                }
            });

            btn_reset.addEventListener('click', () => {
                isRunning = false; nxcPhase = 0; power = 0; actualMotorPower = 0; nxcTimer = 0;
                robotX = START_X; velocity = 0; accel = 0; waterAngle = 0; waterAngularVel = 0;
                isSpilled = false; particles = [];
                
                btn_play.innerText = "▶ Ejecutar Codigo NXC";
                btn_play.style.backgroundColor = "#0d6efd";
                btn_play.style.color = "#fff";
                drawSim();
            });

            slider_wait.addEventListener('input', () => {
                lbl_wait.innerText = slider_wait.value + " ms";
            });
            
            requestAnimationFrame(animate);
        }
        
        initSimulation();
    })();
</script>
"""

html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))


## 1. Guía del Estudiante

### El Desafío
Debes transportar un vaso plástico con agua (o una pelota de ping-pong sobre una superficie plana) desde el punto A al punto B sin derramar una sola gota. Si programas el robot para avanzar y detenerse de golpe, la inercia causará un desastre. Tu misión es programar una **curva de aceleración y frenado suave**.

### Fase 1: Ingeniería (Construcción)
1. Arma la **Base Motriz Estándar** con los motores conectados a los **puertos B y C**.
2. Construye una "cama plana" o plataforma en la parte superior del ladrillo NXT usando vigas de LEGO.
3. Coloca encima un vaso plástico con un poco de agua (¡lejos de los cables!) o una pelota de ping-pong. **Regla de oro:** No puedes pegar ni amarrar la carga al robot.

### Fase 2: Programación (Código NXC)
Si usamos el comando clásico `OnFwd(OUT_BC, 100);` el robot dará un tirón y la carga saldrá volando. 
Para resolver esto, usaremos una nueva herramienta de programación llamada **Bucle `for` (Para)**. Esto nos permite subir la potencia del motor poco a poco, como si estuviéramos pisando el acelerador suavemente.

Copia el siguiente código en BricxCC y lee los comentarios:

```c
task main() {
    int potencia; // Creamos una variable para guardar la fuerza del motor

    // 1. ACELERACIÓN SUAVE (Pisando el acelerador poco a poco)
    // La potencia empieza en 0, sube de 5 en 5, hasta llegar a 70.
    for (potencia = 0; potencia <= 70; potencia = potencia + 5) {
        OnFwd(OUT_BC, potencia); // Enviamos la potencia actual a los motores
        Wait(100);               // Esperamos 0.1 segundos antes de volver a subirla
    }

    // 2. VELOCIDAD CRUCERO (Viaje en carretera)
    // Mantenemos la velocidad constante de 70 por 2 segundos.
    Wait(2000);

    // 3. FRENADO SUAVE (Pisando el freno poco a poco)
    // La potencia empieza en 70, baja de 5 en 5, hasta llegar a 0.
    for (potencia = 70; potencia >= 0; potencia = potencia - 5) {
        OnFwd(OUT_BC, potencia); 
        Wait(100);               // Esperamos 0.1 segundos antes de volver a bajarla
    }

    // Apagamos los motores por seguridad al final
    Off(OUT_BC);
}
```

### Fase 3: Pruebas y Reflexión
* **Compila y descarga** el programa.
* Pon el robot en el suelo con tu carga y ejecútalo. 
* **Experimento y Reflexión:** En el código, cambia el tiempo de espera dentro de los bucles `for` (modifica el `Wait(100);` a `Wait(10);`). Vuelve a compilar. ¿Qué ocurre con el agua o la pelota ahora? ¿Por qué crees que el tiempo de espera en el código afecta directamente a la fuerza de inercia?

---

## 2. Guía del Docente

> **Nota para el Profesor:** Esta actividad transforma un concepto estático (las Leyes de Newton) en un problema de diseño tecnológico. Es ideal para que los estudiantes entiendan que el código no solo mueve cosas, sino que controla **cómo** se mueven esas cosas en el mundo físico.

### Vínculo Curricular (MINEDUC Chile)
* **Ciencias Naturales (7º Básico):** OA 12 - Demostrar, por medio de la investigación experimental, que las fuerzas pueden causar cambios en la forma y/o estado de movimiento de un cuerpo (Inercia).
* **Tecnología (7º Básico):** OA 3 - Comunicar el diseño, la planificación u otros procesos de la creación de productos tecnológicos (Diseño de soluciones para control de movimiento).
* **Matemáticas (8º Básico):** Análisis de secuencias y progresiones aritméticas (el aumento lineal de la variable en el bucle `for`).

### Conceptos Clave a Desarrollar
1. **Primera Ley de Newton (Inercia):** Todo cuerpo preserva su estado de reposo o movimiento uniforme a menos que actúe una fuerza sobre él. Cuando el robot acelera, el vaso de agua quiere quedarse en su lugar (reposo), por lo que se derrama hacia atrás. 
2. **Aceleración vs. Velocidad:** La inercia no se percibe cuando vamos rápido (fase de velocidad crucero), sino cuando **cambiamos** la velocidad (aceleración y frenado). 
3. **El Bucle `for` (Control Estructural):** Es una excelente introducción a los ciclos iterativos. Los alumnos pueden ver visual y físicamente qué significa que una variable se incremente (`potencia = potencia + 5`).

### Resolución de Problemas (Troubleshooting)
* **Los alumnos amarran el vaso:** Asegúrese de que la plataforma sea lisa. Si amarran la carga o le ponen cinta adhesiva, están eludiendo el desafío físico y el experimento de la inercia perderá su propósito.
* **El robot no avanza al principio de la rampa:** Los motores del NXT necesitan al menos un 15% o 20% de potencia para vencer su propio roce interno y el peso del robot. Es normal que en los primeros ciclos del bucle (potencia 0, 5, 10) el robot haga ruido pero no se mueva. Esto permite introducir el concepto de **fuerza de roce estático**.
* **Modificación del nivel de dificultad:** Pida a los estudiantes que transporten la pelota de ping-pong subiendo por un plano inclinado (una rampa de cartón). Tendrán que modificar los parámetros del bucle para que el robot tenga la fuerza necesaria para subir sin botar la pelota.